## **Topics Modeling**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import string
from collections import Counter
from nltk.tokenize import word_tokenize,sent_tokenize,RegexpTokenizer
from nltk.corpus import stopwords,wordnet
from nltk.stem import PorterStemmer,WordNetLemmatizer,SnowballStemmer
from nltk.probability import FreqDist
from nltk import pos_tag
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.util import ngrams,bigrams
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer


In [2]:
# # Load and access Skytrax Airline Review zipped data file using github raw link


# # GitHub raw ZIP file URL
url = "https://raw.githubusercontent.com/nyagayamonica/Skytrax_Airline_Review_Data/main/Skytrax%20Airline%20Review%20Data.zip"

# # Read ZIP file directly
df_airline = pd.read_csv(url, compression="zip")

### **Use Skytrax Dataset to Discover Topics**

The Skytrax Airline Reviews dataset provide a rich context for discovering topics. A passenger may describe a delayed flight, poor communication, uncomfortable seats and an unsatisfactory refund process in a single review. This gives a topic modeling algorithm more information about how different aspects of the airline experience relate to each other.

The dataset is also airline-specific and it creates continuity from the recommendation classifier.
By discovering complaint topics from the same reviews, we can examine relationships between topics, sentiment and passenger recommendations.

After identifying and standardizing topics from the reviews, we will examine whether the same categories accurately describe shorter, less structured airline customer-service messages as on the twitter dataset.

#### **Identify complaint reviews in the Skytrax dataset**

We may discover topics primarily from complaint-oriented reviews, rather than positive reviews. We assume that positive reviews have short phrases that may not be quite useful. e.g good food, comfortable, etc.

Therefore, we will at first use recommended == "no" as a proxy for dissatisfied passengers.

In [29]:
# Select reviews from passengers who did not recommend
# the airline

df_complaints = df_airline[
    df_airline["recommended"].astype(str)
    .str.strip().str.lower() == "no"
].copy()

df_complaints = df_complaints.dropna(
    subset=["review_text"]
)

df_complaints = df_complaints[
    df_complaints["review_text"].astype(str).str.strip() != ""
]

print("Potential complaint reviews:", len(df_complaints))

display(df_complaints["review_text"].head())


Potential complaint reviews: 39848


,review_text
0,My afternoon flight was delayed and eventuall...
1,The flight was postponed 3 times with insuffi...
2,They will charge you for your hand baggage if...
3,The flight was delayed from 15:20 to 16:25. I...
4,No crowd control and boarding was like cattle...


#### **Apply TF-IDF to identify important words and phrases within negative airline reviews.**

In [30]:
tfidf_topics = TfidfVectorizer(
    stop_words="english",
    max_features=3000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90
)

X_topics = tfidf_topics.fit_transform(
    df_complaints["review_text"].astype(str)
)

feature_names = tfidf_topics.get_feature_names_out()

scores = np.asarray(
    X_topics.mean(axis=0)
).ravel()

top_terms = pd.DataFrame({
    "term": feature_names,
    "tfidf_score": scores
}).sort_values(
    "tfidf_score",
    ascending=False
)

display(top_terms.head(12))


,term,tfidf_score
1129,flight,0.062748
2436,service,0.029297
194,airline,0.027567
2701,time,0.026381
1439,hours,0.025082
2390,seat,0.022392
2550,staff,0.021951
2048,plane,0.021655
208,airport,0.021555
203,airlines,0.021313


#### **Apply topic modeling using Non-negative Matrix Factorization (NMF)**

**NMF** works well with TF-IDF representations and is relatively lightweight.

For start, we will request 12 candidate topics, then inspect them to decide how many meaningful categories exist.

In [31]:

from sklearn.decomposition import NMF

# Define the number of topics
n_topics = 12

# Initialize NMF model
nmf_model = NMF(
    n_components=n_topics,
    random_state=42,
    init="nndsvda",
    max_iter=400
)

# Fit NMF model
W = nmf_model.fit_transform(X_topics)

# Extract topic-term matrix
H = nmf_model.components_

# Assign dominant topic to each review
assert W.shape[0] == len(df_complaints), (
    "NMF matrix and DataFrame row counts do not match."
)

df_complaints["dominant_topic"] = W.argmax(axis=1) + 1


#### **Extract the most important keywords associated with each NMF topic.**

In [32]:
# Extract TF-IDF feature names
nmf_feature_names = tfidf_topics.get_feature_names_out()

# Create topic summary
topic_summary = []

for topic_idx, topic_weights in enumerate(nmf_model.components_):

    top_word_indices = topic_weights.argsort()[-10:][::-1]

    top_words = [
        nmf_feature_names[i]
        for i in top_word_indices
    ]

    topic_summary.append({
        "NMF_Topic": topic_idx + 1,
        "Top_Keywords": ", ".join(top_words)
    })

# Convert to DataFrame
df_topic_summary = pd.DataFrame(topic_summary)

display(df_topic_summary)


,NMF_Topic,Top_Keywords
0,1,"flight, hours, delayed, connecting, flight del..."
1,2,"seat, seats, row, flight, paid, extra, economy..."
2,3,"refund, ticket, cancelled, money, told, email,..."
3,4,"bag, check, bags, baggage, carry, checked, pay..."
4,5,"food, crew, cabin, meal, cabin crew, staff, go..."
5,6,"class, business, business class, economy, loun..."
6,7,"american, airlines, american airlines, fly ame..."
7,8,"customer, service, customer service, airline, ..."
8,9,"air, canada, air canada, india, air india, tor..."
9,10,"gate, plane, minutes, boarding, told, said, st..."


#### **Add descriptive NMF topic names**

In [34]:
# Create descriptive topic names from the displayed NMF topics
nmf_topic_names = {
    1: "Flight Delays",
    2: "Seating Issues",
    3: "Refunds and Cancellations",
    4: "Grounds Service",
    5: "In-flight Experience",
    6: "Cabin Class and Upgrades",
    7: "Airline-Specific Discussions",
    8: "Customer Service",
    9: "Airline-Specific Discussions",
    10: "Boarding and Gate Operations",
    11: "Lost Baggage",
    12: "Airline-Specific Discussions"
}

df_complaints["nmf_topic_name"] = (
    df_complaints["dominant_topic"].map(nmf_topic_names)
)

display(
    df_complaints[
        ["review_text", "dominant_topic", "nmf_topic_name"]
    ].head(20)
)


,review_text,dominant_topic,nmf_topic_name
0,My afternoon flight was delayed and eventuall...,1,Flight Delays
1,The flight was postponed 3 times with insuffi...,9,Airline-Specific Discussions
2,They will charge you for your hand baggage if...,4,Grounds Service
3,The flight was delayed from 15:20 to 16:25. I...,9,Airline-Specific Discussions
4,No crowd control and boarding was like cattle...,4,Grounds Service
5,30 hours till 6.30 PM evening. I tried to call...,1,Flight Delays
7,They have changed my seat twice first given m...,2,Seating Issues
8,Ground check in staff was pathetic. First of ...,4,Grounds Service
9,"Incident Overview: Upon boarding the flight, ...",8,Customer Service
10,"However, the airline cancelled our flight and...",1,Flight Delays


#### **Inspect actual reviews associated with each topic**

In [38]:
# Actual reviews associated with each topic
for topic_id in range(1, n_topics + 1):

    topic_index = topic_id - 1

    top_positions = np.argsort(
        W[:, topic_index]
    )[-3:][::-1]

    print(f"\n{'=' * 60}")
    print(f"TOPIC {topic_id}: {nmf_topic_names[topic_id]}")
    print("=" * 60)

    for pos in top_positions:
        review = df_complaints.iloc[pos]["review_text"]
        print(str(review)[:500])
        print("-" * 60)



TOPIC 1: Flight Delays
 First flight was delayed 2 hours in advance, automatically rebooked to a flight on the same day, 5 hours later. This flight was then delayed two times, half an hour before scheduled departure time with a total of 4 more hours. Missed the connecting flight, and of course the "help desk" cannot do anything and a refund or compensation is pure myth.
------------------------------------------------------------
 I missed my scheduled flight and had to rebook, the representative at the airport put me on standby for a flight several hours later, telling me that I would "probably get on." However, the flight was full, and after waiting around for over three hours I rebooked for the following day. I arrived at the airport two hours early this time, only to find my first flight had been delayed by two hours. I received a message on my phone that this might cause me to miss my connection in Chicago, so I att
------------------------------------------------------------
 Th

## **Standardizing Airline Complaint Topics and Sentiment Using Ollama**

We will use Ollama as an automated labeling tool to classify customer tweets into standardized airline complaint topics and sentiment categories, without manually labeling the entire tweets dataset.

Our objective is to create a consistent labeling framework that works across both datasets:
- Skytrax airline reviews: Longer, detailed passenger experiences.
- Twitter customer-support dataset: Short customer messages, including complaints, questions, praise, and conversations with company representatives.

We will use Ollama to generate labels, then evaluate their quality and eventually use those labels to train a faster classification model.

In [46]:
# Install zstd package to help extract Ollama Installation files
!apt-get update -qq
!apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Install Python libraries
!pip install -q ollama pydantic

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/noble/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.189.187), connection timed out Could not connect to ppa.launchpadcontent.net:443 (185.125.189.186), connection timed out Could not connect to ppa.launchpadcontent.net:443 (185.125.189.188), connection timed out
W: Some index files failed to download. They have been ignored, or old ones used instead.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 40 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#

In [47]:
# # Start the Ollama server (#For google colab)
# # Run the code below

import subprocess
import time
import urllib.request

def ollama_running():
    try:
        urllib.request.urlopen(
            "http://localhost:11434/api/tags",
            timeout=2
        )
        return True
    except Exception:
        return False

if not ollama_running():
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    for _ in range(30):
        if ollama_running():
            break
        time.sleep(1)

if not ollama_running():
    raise RuntimeError("Ollama server failed to start.")

print("Ollama server is running.")


Ollama server is running.


In [48]:
# Download Ollama model


import subprocess
from ollama import Client

client = Client(host="http://localhost:11434")

MODEL_NAME = "qwen2.5:1.5b"

installed_models = [
    model.model
    for model in client.list().models
]

if MODEL_NAME not in installed_models:
    print("Downloading model...")
    subprocess.run(
        ["ollama", "pull", MODEL_NAME],
        check=True
    )
else:
    print("Model already available.")

print("Ollama setup completed.")


Model already available.
Ollama setup completed.


In [51]:
# Import Ollama Libraries
import ollama
import json
from ollama import Client
from pydantic import BaseModel
from typing import Literal

client = Client(host="http://localhost:11434")
MODEL_NAME = "qwen2.5:1.5b"

# Test the connection
response = client.chat(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Ollama is working"
        }
    ]
)

print(response.message.content)

Ollama is working


#### **Create the Qwen classification prompt**

We will use a labeling framework that both the Skytrax and Twitter datasets will follow.

In [64]:

SYSTEM_PROMPT = """
You are an expert airline passenger feedback analyst.

Your task is to analyze airline passenger reviews
and classify their complaint topics and sentiment.

The complaint taxonomy is based on 12 topics discovered
through Non-negative Matrix Factorization (NMF)
on airline passenger reviews.

STANDARDIZED COMPLAINT CATEGORIES:

1. DELAYS
NMF Topic 1:
Keywords: flight, hours, delayed, connecting,
connecting flight, airport, hotel, delay, missed.

Description:
Complaints about delayed flights, long waiting times,
missed connections, and flight disruptions.

Standard label: Delays

2. SEATING
NMF Topic 2:
Keywords: seat, seats, row, paid, extra, economy,
leg, middle, sit, aisle, window.

Description:
Complaints about uncomfortable seats, limited legroom,
seat allocation, seating arrangements, and extra
seat charges.

Standard label: Seating

3. REFUNDS
NMF Topic 3:
Keywords: refund, ticket, cancelled, money, email,
change, credit, days, tickets, airline.

Description:
Complaints about refund requests, reimbursement,
compensation, and delayed repayment.

Standard label: Refunds

Note: Ticket changes and reservation issues without
a refund dispute belong to Booking & Boarding.

4. BAGGAGE
NMF Topics 4 and 11:
Keywords: bag, check, bags, baggage, carry, checked,
pay, extra, weight, charged, luggage, lost, days,
claim, arrived.

Description:
Complaints about lost, delayed or damaged baggage,
baggage charges, baggage restrictions, and luggage
handling.

Standard label: Baggage

5. FOOD & BEVERAGE
NMF Topic 5:
Keywords: food, crew, cabin, meal, staff, good,
entertainment, service, water, served.

Description:
Complaints about food quality, meal availability,
beverages, and in-flight catering.

Standard label: Food & Beverage

Note: Crew behavior belongs to Customer Service,
while entertainment problems belong to Entertainment.

6. SEATING AND CABIN CLASS
NMF Topic 6:
Keywords: class, business, business class, economy,
lounge, premium, upgrade, class seats.

Description:
Complaints about cabin comfort, seating quality,
seat upgrades, and class-related seating expectations.

Standard label: Seating

Note: Lounge problems or other cabin-class complaints
that do not concern seating should be assigned to
Other or another relevant category.

7. CUSTOMER SERVICE
NMF Topic 8:
Keywords: customer, service, worst, rude, help,
poor, terrible, care, phone.

Description:
Complaints about rude staff, poor communication,
unhelpful support, and customer care.

Standard label: Customer Service

8. BOOKING & BOARDING
NMF Topics 3 and 10:
Keywords: ticket, cancelled, change, gate, plane,
minutes, boarding, told, staff, passengers, board.

Description:
Complaints about reservations, ticket changes,
check-in, boarding procedures, and gate operations.

Standard label: Booking & Boarding

Note: Flight departure delays and missed connections
belong to Delays. Staff behavior belongs to
Customer Service.

9. ENTERTAINMENT
NMF Topic 5:
Keywords: entertainment, cabin, service, flight.

Description:
Complaints about in-flight entertainment systems,
movies, screens, Wi-Fi, and entertainment availability.

Standard label: Entertainment

10. AIRLINE-SPECIFIC TOPICS
NMF Topics 7, 9 and 12:
Keywords include:
American Airlines, Turkish Airlines, Air Canada,
Air India, Air France, United Airlines, Delta.

These topics represent airline names rather than
complaint categories.

IMPORTANT:
Never assign an airline name as a complaint topic.
Read the actual review and identify the underlying
service issue using the standardized categories.

11. OTHER
Use Other when a genuine complaint does not fit
any of the defined categories.

12. NO COMPLAINT
Use No Complaint when the review contains no
clear dissatisfaction or service problem.

ALLOWED PRIMARY AND SECONDARY TOPIC LABELS:

- Delays
- Seating
- Refunds
- Baggage
- Food & Beverage
- Customer Service
- Booking & Boarding
- Entertainment
- Other
- No Complaint

SENTIMENT CLASSIFICATION:

Positive:
The passenger expresses overall satisfaction,
praise, or a favorable experience.

Neutral:
The passenger provides factual information,
has unclear emotional tone, or expresses
approximately balanced positive and negative views.

Negative:
The passenger expresses dissatisfaction,
frustration, disappointment, or criticism.

CLASSIFICATION RULES:

1. Analyze the actual review text, not just NMF keywords.

2. Assign exactly one primary_topic.

3. Assign all clearly expressed complaint categories
   to all_topics.

4. If multiple complaints exist, choose the most
   emphasized issue as the primary topic.

5. If two complaints are equally emphasized,
   choose the one mentioned first.

6. Do not invent complaints not supported by the text.

7. Do not classify airline names as complaint topics.

8. A review may contain multiple complaint topics.

9. If no complaint exists, assign No Complaint
   as primary_topic and the only entry in all_topics.

10. Assign exactly one overall sentiment:
    Positive, Neutral, or Negative.

11. Sentiment must reflect the entire review,
    including positive and negative statements.

12. Keep the explanation short and evidence-based.

13. Return ONLY valid JSON using this structure:

{
    "primary_topic": "one allowed topic",
    "all_topics": ["one or more allowed topics"],
    "sentiment": "Positive, Neutral, or Negative",
    "explanation": "brief explanation"
}

Do not include Markdown or additional text.
"""


In [65]:
# Define the output structure

TopicLabel = Literal[
    "Delays",
    "Seating",
    "Refunds",
    "Baggage",
    "Food & Beverage",
    "Customer Service",
    "Booking & Boarding",
    "Entertainment",
    "Other",
    "No Complaint"
]

class ComplaintClassification(BaseModel):

    primary_topic: TopicLabel

    all_topics: list[TopicLabel]

    sentiment: Literal[
        "Positive",
        "Neutral",
        "Negative"
    ]

    explanation: str


In [67]:
#  Create the classification function

client = Client(host="http://localhost:11434")

MODEL_NAME = "qwen2.5:1.5b"

def classify_complaint(review_text):

    response = client.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": (
                    "Classify the following airline review:\n\n"
                    + str(review_text)
                )
            }
        ],
        format=ComplaintClassification.model_json_schema(),
        options={
            "temperature": 0
        }
    )

    result = ComplaintClassification.model_validate_json(
        response.message.content
    )

    return result.model_dump()



In [68]:
# Test on random reviews
test_reviews = [
    "My flight was delayed for six hours and I missed my connection.",

    "The airline lost my luggage and customer service refused to help.",

    "I requested a refund after my flight was cancelled but never received my money.",

    "The business class seats were uncomfortable and the food was terrible.",

    "I flew with United Airlines and the staff were extremely rude.",

    "The flight was excellent, the crew were friendly and the food was delicious."
]

for review in test_reviews:

    result = classify_complaint(review)

    print("\nREVIEW:", review)
    print("PRIMARY TOPIC:", result["primary_topic"])
    print("ALL TOPICS:", result["all_topics"])
    print("SENTIMENT:", result["sentiment"])
    print("EXPLANATION:", result["explanation"])
    print("-" * 60)



REVIEW: My flight was delayed for six hours and I missed my connection.
PRIMARY TOPIC: Delays
ALL TOPICS: ['Delays']
SENTIMENT: Negative
EXPLANATION: The review mentions a flight delay for six hours, which is a clear complaint about delays.
------------------------------------------------------------

REVIEW: The airline lost my luggage and customer service refused to help.
PRIMARY TOPIC: Baggage
ALL TOPICS: ['Baggage', 'Customer Service']
SENTIMENT: Negative
EXPLANATION: The review mentions both a lost luggage issue and a lack of customer service, indicating a negative overall experience.
------------------------------------------------------------

REVIEW: I requested a refund after my flight was cancelled but never received my money.
PRIMARY TOPIC: Refunds
ALL TOPICS: ['Refunds']
SENTIMENT: Negative
EXPLANATION: The review explicitly mentions a refund request and the failure to receive the refund, indicating a negative sentiment towards the refund process.
-------------------------